## Setup

Keep [`example_data.py`](example_data.py), [`data_fidelity_metrics.py`](data_fidelity_metrics.py), and [`missingness_fidelity_metrics.py`](missingness_fidelity_metrics.py) beside this notebook. The project environment supplies NumPy, pandas, Matplotlib, and IPython.

In [ ]:
#| label: setup

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from data_fidelity_metrics import hellinger_distance
from example_data import make_example_views
from missingness_fidelity_metrics import (
    joint_missingness_details,
    missingness_hellinger_distance,
)

## Learning goals

By the end of this lesson, you should be able to:

- read a row-wise missingness pattern $M_i^X$ or $M_i^Y$;
- calculate the empirical probability of each pattern;
- align the patterns observed in $X$ and $Y$;
- calculate their joint missingness Hellinger distance; and
- explain why matching marginal rates does not guarantee matching joint patterns.

## 1. Metric, inputs, and notation

The **Joint Missingness Hellinger Distance** is a missingness-fidelity metric. We write it as $\mathrm{Hel}_{\mathrm{miss}}^{\mathrm{joint}}$. It compares which variables are missing together in the same record.

We use the same notation as the previous lesson:

- $X$ is the reference dataset, with dimensions $n_X\times d$.
- $Y$ is the comparison dataset, with dimensions $n_Y\times d$.
- $n_X$ and $n_Y$ are the numbers of records in $X$ and $Y$, and $d$ is the number of variables shared by both datasets.
- $M^X$ and $M^Y$ are the missingness indicator matrices, with dimensions $n_X\times d$ and $n_Y\times d$.
- $M_i^X$ and $M_i^Y$ are the missingness patterns of record $i$ in $X$ and $Y$.

This metric uses the original incomplete datasets. It does not use $\widetilde X$ or $\widetilde Y$. In the shared example, $d=4$, $n_X=6$, and $n_Y=5$.

In [ ]:
#| label: missingness-inputs

example = make_example_views()
X, Y = example.X, example.Y
M_X, M_Y = example.M_X, example.M_Y

print("Variable order:", list(X.columns))
print(f"d = {X.shape[1]}, n_X = {X.shape[0]}, n_Y = {Y.shape[0]}")

## 2. Row-wise missingness patterns

A row of $M^X$ is the complete missingness pattern of one record:

$$
M_i^X=
\left((M^X)_{i1},\ldots,(M^X)_{id}\right)
\in\{0,1\}^d
$$

The pattern $M_i^Y$ is defined in the same way from $M^Y$. Each position follows the shared variable order. A $0$ means observed. A $1$ means missing.

With $d$ variables, there are at most $2^d$ possible patterns. We only need the patterns observed in at least one table.

In [ ]:
#| label: row-missingness-patterns

M_X_patterns = M_X.copy()
M_X_patterns["M_i^X"] = M_X.apply(tuple, axis=1)
M_Y_patterns = M_Y.copy()
M_Y_patterns["M_i^Y"] = M_Y.apply(tuple, axis=1)

print("Rows of M^X")
display(M_X_patterns)
print("Rows of M^Y")
display(M_Y_patterns)

## 3. Align the pattern probabilities

Let $\mathcal S$ be the union of the patterns observed in $X$ and $Y$. For each pattern $s\in\mathcal S$, define its empirical probabilities as

$$
p_s^X=\frac1{n_X}\sum_{i=1}^{n_X}
\mathbf 1\{M_i^X=s\},
\qquad
p_s^Y=\frac1{n_Y}\sum_{i=1}^{n_Y}
\mathbf 1\{M_i^Y=s\}
$$

A pattern that is absent from one dataset receives probability $0$ in that dataset. The aligned vectors $\mathbf p^X=(p_s^X)_{s\in\mathcal S}$ and $\mathbf p^Y=(p_s^Y)_{s\in\mathcal S}$ each sum to $1$.

In [ ]:
#| label: aligned-pattern-probabilities

details = joint_missingness_details(X, Y)
display(details.round(4))
print(f"sum p_X = {details['P_X(pattern)'].sum():.1f}")
print(f"sum p_Y = {details['P_Y(pattern)'].sum():.1f}")

## 4. Joint missingness Hellinger distance

The metric is the Hellinger distance between the two aligned pattern distributions:

$$
\mathrm{Hel}_{\mathrm{miss}}^{\mathrm{joint}}(X,Y)=
\sqrt{\frac12\sum_{s\in\mathcal S}
\left(\sqrt{p_s^X}-\sqrt{p_s^Y}\right)^2}
$$

A value of $0$ means the two datasets have the same pattern distribution. A small value means their pattern distributions are similar. A large value means they are different. A value of $1$ means the two datasets have no observed missingness pattern in common.

Unlike the marginal metric, this calculation produces one Hellinger distance. It does not calculate one distance per variable and then average.

In [ ]:
#| label: calculate-joint-missingness

p_X_patterns = details["P_X(pattern)"].to_numpy()
p_Y_patterns = details["P_Y(pattern)"].to_numpy()
Hel_miss_joint_from_vectors = hellinger_distance(p_X_patterns, p_Y_patterns)
Hel_miss_joint = missingness_hellinger_distance(X, Y, mode="joint")

print(f"Joint missingness Hellinger distance = {Hel_miss_joint:.6f}")
assert np.isclose(Hel_miss_joint, Hel_miss_joint_from_vectors)

In [ ]:
#| label: fig-joint-missingness
#| fig-cap: "Empirical probabilities of the aligned row-wise missingness patterns."

pattern_labels = ["".join(map(str, pattern)) for pattern in details["pattern"]]
positions = np.arange(len(details))
width = 0.36

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    positions - width / 2,
    details["P_X(pattern)"],
    width,
    label="X",
)
ax.bar(
    positions + width / 2,
    details["P_Y(pattern)"],
    width,
    label="Y",
)
ax.set(
    xlabel="Missingness pattern",
    ylabel="Probability",
    ylim=(0, 1),
    xticks=positions,
    xticklabels=pattern_labels,
)
ax.legend()
plt.show()

## 5. Why the joint metric is different

Two datasets can have the same missingness rate for every variable but different row patterns. The next example has two variables. Each variable is missing in half of the records in both datasets. Therefore, the marginal distance is $0$.

In the reference data, exactly one variable is missing in each record. In the comparison data, either both variables are observed or both are missing. The two sets of row patterns do not overlap, so the joint distance is $1$.

In [ ]:
#| label: same-marginals-different-joint-patterns

X_demo = pd.DataFrame(
    {
        "a": [np.nan, 1.0],
        "b": [1.0, np.nan],
    }
)
Y_demo = pd.DataFrame(
    {
        "a": [1.0, np.nan],
        "b": [1.0, np.nan],
    }
)
Hel_miss_marg_demo = missingness_hellinger_distance(
    X_demo, Y_demo, mode="marginal"
)
Hel_miss_joint_demo = missingness_hellinger_distance(
    X_demo, Y_demo, mode="joint"
)

print("M^X_demo")
display(X_demo.isna().astype(int))
print("M^Y_demo")
display(Y_demo.isna().astype(int))
print(f"Marginal distance = {Hel_miss_marg_demo:.1f}")
print(f"Joint distance = {Hel_miss_joint_demo:.1f}")
assert np.isclose(Hel_miss_marg_demo, 0.0)
assert np.isclose(Hel_miss_joint_demo, 1.0)

In [ ]:
#| label: metric-sanity-checks

Hel_miss_joint_same = missingness_hellinger_distance(X, X, mode="joint")
Hel_miss_joint_reversed = missingness_hellinger_distance(Y, X, mode="joint")

print(f"Hel_miss_joint(X, X) = {Hel_miss_joint_same:.6f}")
print(f"Hel_miss_joint(Y, X) = {Hel_miss_joint_reversed:.6f}")
assert np.isclose(Hel_miss_joint_same, 0.0)
assert np.isclose(Hel_miss_joint, Hel_miss_joint_reversed)
assert 0.0 <= Hel_miss_joint <= 1.0

## 6. Interpretation

For the shared example, $\mathrm{Hel}_{\mathrm{miss}}^{\mathrm{joint}}(X,Y)\approx0.0712$. This is a small value. The two datasets contain the same four patterns with similar probabilities.

- A value of $0$ means the pattern distributions are identical.
- A small value means the row-wise missingness patterns have similar probabilities.
- A large value means the pattern probabilities are different.
- A value of $1$ means the observed pattern sets do not overlap.
- The value is not the percentage of records with different patterns.
- The metric is symmetric. Swapping $X$ and $Y$ does not change it.

The joint metric captures more structure than the marginal metric. However, rare patterns can be unstable in small datasets. The number of possible patterns also grows as $2^d$, so large $d$ can produce many sparse patterns.

## Takeaway

The Joint Missingness Hellinger Distance follows four steps:

1. build $M^X$ and $M^Y$;
2. treat each row $M_i^X$ or $M_i^Y$ as one pattern;
3. align the pattern probabilities across $X$ and $Y$; and
4. calculate one Hellinger distance between the two distributions.

A smaller result means that $X$ and $Y$ have more similar row-wise missingness patterns.